# 00c · 조건부 기대, pgf, 지수분포 (Conditional Expectation, pgf, and the Exponential Distribution)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Ross 12e §3.1–3.4 (조건부 기대·조건화로 기대와 분산 계산·랜덤 합), §3.5 (조건화로 확률 계산), §5.2.1–5.2.3 (지수분포: 무기억성, 최솟값, $P(X_1<X_2)$) · Durrett 3e §2.1 (지수분포), §2.3 (복합 푸아송), §5.1 (조건부 기대) · G&S 4e §3.7 (조건부 분포·기대), §5.1 (생성함수, 랜덤 합 $G_S=G_N\circ G_Y$), §5.2 |
| 선수 노트북 | 00a, 00b |
| 예상 소요 | 90분 |
| 상태 | draft |

이 노트북은 앞으로 모든 모듈이 쓸 확률 도구 세 개를 시뮬레이션과 함께 손에 넣는 시간이다. (1) **조건화(conditioning)**: $E[X\mid Y]$는 $Y$의 함수인 확률변수이고, 탑 성질과 분산 분해가 첫걸음 분석(00a)·마팅게일(05)의 언어다. (2) **랜덤 합과 pgf**: 분기 과정(01)·복합 푸아송(02)의 분산이 여기서 나온다. (3) **지수분포의 경주**: 율은 더해지고 승자는 율에 비례하며 시각과 독립 — 푸아송 과정(02)과 CTMC(03)의 심장이다.

## 0. 준비 (Setup)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from scipy.stats import gamma
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.poisson import compound_poisson
from spkit.plots import plot_hist_vs_pmf, plot_hist_vs_pdf

SEED, rng = rng_for("00c")
setup_plots()
N_PATHS = scale(200_000)  # FAST(SPKIT_FAST=1) 모드에서는 1/10 크기
n = N_PATHS               # 이 노트북은 i.i.d. 표본 크기를 n 으로 쓴다
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답을 적어 본 뒤 정답을 펼쳐 보세요.

**Q1.** (00b) MC 추정량의 SE 공식은? 표본을 4배로 늘리면 95% 구간의 폭은 몇 배가 되나?

<details><summary>정답</summary>

$s/\sqrt n$ (베르누이는 $\sqrt{\hat p(1-\hat p)/n}$). 폭은 $1/\sqrt4=1/2$배.

</details>

**Q2.** (00b) '95% 신뢰구간'의 포함률이 정확히 0.95가 아니었던 예와 그 이유는?

<details><summary>정답</summary>

$p=0.05$, $n=100$의 Wald 구간은 포함률 0.877. $np=5$뿐이라 정규 근사가 나쁘고, $\hat p=0$이면 (보정 전) 구간 폭이 0이 된다.

</details>

**Q3.** (00b) `assert_close(k=4)`가 FAST 모드에서도 유효한 이유는?

<details><summary>정답</summary>

허용폭이 SE의 4배로 정의되어 표본이 줄어 SE가 커지면 허용폭도 함께 커진다. 거짓 경보 확률(약 $6\times10^{-5}$)이 표본 크기와 무관하다.

</details>

**Q4 (연결 고리).** (00a) 파산 확률 $h(i)$가 만족하는 첫걸음 방정식은? 그 방정식이 성립하려면 걸음에 어떤 성질이 필요했나?

<details><summary>정답</summary>

$h(i)=p\,h(i+1)+q\,h(i-1)$. 걸음의 독립·동일분포(미래가 현재 위치만으로 결정). 첫걸음으로 **조건화**해서 확률을 계산한 것인데, 오늘 배우는 조건부 기대가 바로 그 논리의 일반형이다.

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **조건부 기대(conditional expectation).** (이산) $E[X\mid Y=y]=\sum_x x\,P(X=x\mid Y=y)$; (연속) $\int x\,f_{X\mid Y}(x\mid y)\,dx$. 함수 $g(y)=E[X\mid Y=y]$를 $Y$에 대입한
  $$E[X\mid Y]:=g(Y)$$
  는 **확률변수**다. 이 노트북 내내 $E|X|<\infty$를 가정한다. (Ross 12e §3.1–3.2, G&S 4e §3.7, Durrett 3e §5.1)
- **탑 성질(tower property).** $E[E[X\mid Y]]=E[X]$. **조건부 분산.** $\mathrm{Var}(X\mid Y)=E[X^2\mid Y]-E[X\mid Y]^2$, 그리고
  $$\mathrm{Var}X=E[\mathrm{Var}(X\mid Y)]+\mathrm{Var}(E[X\mid Y]).$$
- **랜덤 합(random sum).** $N\in\{0,1,2,\dots\}$, $Y_1,Y_2,\dots$ i.i.d., $N$과 $(Y_i)$가 독립일 때 $S=\sum_{i=1}^N Y_i$ ($N=0$이면 $S=0$). $N\sim\mathrm{Poisson}(\lambda)$이면 **복합 푸아송(compound Poisson)**. (Ross 12e §3.4, Durrett 3e §2.3)
- **확률생성함수(probability generating function, pgf).** 비음 정수값 $X$에 대해 $G_X(s)=E[s^X]=\sum_k p_k s^k$, $|s|\le1$. $G(1)=1$, $G'(1)=E[X]$, $G''(1)=E[X(X-1)]$, $p_k=G^{(k)}(0)/k!$; 독립이면 $G_{X+Y}=G_XG_Y$. **단위원 위 값에서 pmf 복원**: $M>$(지지집합 크기)이면
  $$p_k=\frac1M\sum_{j=0}^{M-1}G\bigl(e^{2\pi ij/M}\bigr)e^{-2\pi ijk/M}\qquad(\text{FFT 한 줄}).$$
  (G&S 4e §5.1)
- **지수분포 $\mathrm{Exp}(\lambda)$.** 밀도 $\lambda e^{-\lambda x}$ ($x>0$), $P(X>t)=e^{-\lambda t}$, $E[X]=1/\lambda$, $\mathrm{Var}X=1/\lambda^2$. numpy: `rng.exponential(scale=1/lambda)` — 인수가 rate가 아니라 **scale(=평균)**이다.
- **무기억성(memorylessness).** 모든 $s,t\ge0$에 대해 $P(X>s+t\mid X>s)=P(X>t)$. 조건부 잔여수명 $(X-s)\mid X>s\sim\mathrm{Exp}(\lambda)$. (Ross 12e §5.2.2)
- **경쟁하는 시계(competing clocks).** 독립 $X_i\sim\mathrm{Exp}(\lambda_i)$, $i=1,\dots,k$, $\Lambda=\sum_i\lambda_i$. $M=\min_iX_i$, $I=\arg\min_iX_i$. (Ross 12e §5.2.3, Durrett 3e §2.1)

### 2.2 정리 A — 탑 성질과 조건부 분산 공식 (Ross 12e §3.4, G&S 4e §3.7)

> $E|X|<\infty$이면 $E[E[X\mid Y]]=E[X]$. $E[X^2]<\infty$이면
> $$\mathrm{Var}X=E[\mathrm{Var}(X\mid Y)]+\mathrm{Var}(E[X\mid Y]).$$

**가정이 왜 필요한가**
- **$E|X|<\infty$.** 증명은 이중합(이중적분)의 순서를 바꾸는 한 줄이 전부인데, 순서 교환(Fubini/Tonelli)에는 절대수렴이 필요하다.
- **$E[X^2]<\infty$ (분산 공식).** 그렇지 않으면 양변이 정의되지 않는다.

<details><summary>증명 (완전 증명, 이산 경우)</summary>

**탑 성질.** $E[X\mid Y]$는 $Y$의 함수 $g(Y)$이므로 그 기댓값은 $Y$의 pmf로 가중한 합이다:
$$E[E[X\mid Y]]=\sum_y P(Y=y)\,g(y)=\sum_y P(Y=y)\sum_x x\,P(X=x\mid Y=y)=\sum_y\sum_x x\,P(X=x,Y=y).$$
$E|X|<\infty$이므로 합의 순서를 바꿀 수 있고, $\sum_x x\sum_y P(X=x,Y=y)=\sum_x x\,P(X=x)=E[X]$. (연속이면 합을 적분으로 바꾸면 된다.)

**분산 공식.** 탑 성질을 $X^2$에 적용하고 조건부 분산의 정의 $E[X^2\mid Y]=\mathrm{Var}(X\mid Y)+E[X\mid Y]^2$를 넣는다:
$$E[X^2]=E\bigl[E[X^2\mid Y]\bigr]=E[\mathrm{Var}(X\mid Y)]+E\bigl[E[X\mid Y]^2\bigr].$$
여기서 $(E[X])^2=\bigl(E[E[X\mid Y]]\bigr)^2$ (탑 성질)를 빼면
$$\mathrm{Var}X=E[\mathrm{Var}(X\mid Y)]+\Bigl\{E\bigl[E[X\mid Y]^2\bigr]-\bigl(E[E[X\mid Y]]\bigr)^2\Bigr\}=E[\mathrm{Var}(X\mid Y)]+\mathrm{Var}(E[X\mid Y]).\ \square$$

</details>

### 2.3 정리 B — 랜덤 합의 평균·분산, 복합 푸아송 (Ross 12e §3.4, Durrett 3e §2.3)

> $N$과 i.i.d. $(Y_i)$가 독립이고 $E[N],E[N^2],E[Y^2]<\infty$이면
> $$E[S]=E[N]\,E[Y],\qquad \mathrm{Var}(S)=E[N]\,\mathrm{Var}(Y)+\mathrm{Var}(N)\,(E[Y])^2.$$
> 특히 $N\sim\mathrm{Poisson}(\lambda)$이면 $\mathrm{Var}(S)=\lambda\,E[Y^2]$.

**가정이 왜 필요한가**
- **$N$과 $(Y_i)$의 독립.** $E[S\mid N=n]=nE[Y]$, $\mathrm{Var}(S\mid N=n)=n\mathrm{Var}(Y)$라고 쓰는 단계에서 필요하다. $N$이 $Y$들을 **보고** 정해지면(E3) 분산 공식이 깨진다. 평균 공식은 $N$이 "과거만 보는" 정지 시간이면 살아남는다(Wald 항등식, 05c).
- **$Y_i$의 i.i.d.** 조건부 분산이 $n\mathrm{Var}(Y)$가 되려면 독립(공분산 0)과 동일분산이 필요하다.
- **2차 모멘트 유한.** 정리 A의 가정이다.

<details><summary>증명 (완전 증명 — 이 노트북의 대표 증명)</summary>

$N=n$으로 조건을 걸면 독립성에 의해 $S$는 i.i.d. $n$개의 합 $Y_1+\dots+Y_n$과 같은 분포를 가진다. 따라서
$$E[S\mid N]=N\,E[Y],\qquad \mathrm{Var}(S\mid N)=N\,\mathrm{Var}(Y)\quad(\text{독립 합의 분산은 분산의 합}).$$
정리 A의 탑 성질: $E[S]=E[N\,E[Y]]=E[N]\,E[Y]$. 정리 A의 분산 공식:
$$\mathrm{Var}S=E[\mathrm{Var}(S\mid N)]+\mathrm{Var}(E[S\mid N])=E[N\,\mathrm{Var}Y]+\mathrm{Var}(N\,EY)=E[N]\,\mathrm{Var}Y+\mathrm{Var}N\,(EY)^2.$$
푸아송이면 $E[N]=\mathrm{Var}N=\lambda$이므로 $\lambda\bigl(\mathrm{Var}Y+(EY)^2\bigr)=\lambda E[Y^2]$. $\square$

**숫자로.** $Y\in\{1,2\}$ 등확률: $EY=1.5$, $\mathrm{Var}Y=0.25$, $E[Y^2]=2.5$. $N\sim\mathrm{Poisson}(3)$: $\mathrm{Var}S=3\cdot2.5=7.5$. $N\sim\mathrm{Bin}(6,\tfrac12)$ (같은 $E[N]=3$, $\mathrm{Var}N=1.5$): $\mathrm{Var}S=3\cdot0.25+1.5\cdot2.25=4.125$.

</details>

### 2.4 정리 C — 랜덤 합의 pgf (G&S 4e §5.1)

> 정리 B의 가정 하에 $Y_i$가 비음 정수값이면 $G_S(s)=G_N\bigl(G_Y(s)\bigr)$.

**가정이 왜 필요한가**
- **$N$과 $(Y_i)$ 독립, $Y_i$ i.i.d.** $E[s^S\mid N=n]=E[s^{Y_1}]\cdots E[s^{Y_n}]=G_Y(s)^n$이 되려면 독립 곱의 기댓값이 기댓값의 곱이어야 한다.

<details><summary>증명 (완전 증명)</summary>

$N$으로 조건을 걸고(탑 성질) 독립성을 쓴다:
$$G_S(s)=E[s^S]=\sum_{n\ge0}P(N=n)\,E\bigl[s^{Y_1+\cdots+Y_n}\bigr]=\sum_{n\ge0}P(N=n)\,G_Y(s)^n=G_N\bigl(G_Y(s)\bigr).$$
일관성 검사: 연쇄법칙으로 $G_S'(1)=G_N'(G_Y(1))\,G_Y'(1)=G_N'(1)\,G_Y'(1)=E[N]\,E[Y]$ — 정리 B의 평균 공식과 같다. $\square$

</details>

### 2.5 정리 D — 지수분포의 경주 (Ross 12e §5.2.3, Durrett 3e §2.1)

> 독립 $X_i\sim\mathrm{Exp}(\lambda_i)$, $\Lambda=\sum_i\lambda_i$이면
> (i) $M=\min_iX_i\sim\mathrm{Exp}(\Lambda)$; (ii) $P(I=i)=\lambda_i/\Lambda$; (iii) $M$과 $I$는 독립: $P(I=i,\,M>t)=(\lambda_i/\Lambda)\,e^{-\Lambda t}$.

**가정이 왜 필요한가**
- **독립성.** $P(M>t)=\prod_iP(X_i>t)$로 곱해지는 근거다. 시계들이 상관되면 (i)–(iii) 모두 깨진다.
- **지수분포(연속).** 동시 발생(tie)의 확률이 0이라 $I$가 잘 정의되고, (iii)의 독립성은 무기억성 때문에 성립한다 — 서로 다른 구간의 균등분포 시계(예: $U(0,1),U(0,\tfrac12),U(0,\tfrac13)$)로 바꾸면 최솟값의 크기와 승자가 종속된다 (동일분포 시계라면 대칭성으로 여전히 독립이다).

<details><summary>증명 (완전 증명)</summary>

(i) 독립이므로 $P(M>t)=\prod_iP(X_i>t)=\prod_ie^{-\lambda_it}=e^{-\Lambda t}$: 꼬리가 $\mathrm{Exp}(\Lambda)$의 꼬리다.

(ii)와 (iii)를 동시에. 사건 $\{I=i,\,M>t\}$는 "$X_i=x>t$이고 나머지는 모두 $x$보다 크다"이므로 $X_i$의 밀도로 적분한다:
$$P(I=i,\,M>t)=\int_t^\infty\lambda_ie^{-\lambda_ix}\prod_{j\ne i}e^{-\lambda_jx}\,dx=\lambda_i\int_t^\infty e^{-\Lambda x}\,dx=\frac{\lambda_i}{\Lambda}e^{-\Lambda t}.$$
$t=0$을 넣으면 $P(I=i)=\lambda_i/\Lambda$ (ii). 그리고 우변이 $P(I=i)\cdot P(M>t)$의 **곱**이므로 $I$와 $M$은 독립 (iii). $\square$

</details>

### 2.6 정리 E — 무기억성의 특성화 (인용: Ross 12e §5.2.2, Durrett 3e §2.1)

> 연속 양의 확률변수 중 무기억성을 만족하는 것은 지수분포뿐이다 (이산에서는 기하분포).

**가정이 왜 필요한가**
- **연속성(또는 이산 격자).** 무기억성은 꼬리 $\bar F(t)=P(X>t)$에 대한 함수방정식 $\bar F(s+t)=\bar F(s)\bar F(t)$이다. 그 해가 지수함수 $e^{-\lambda t}$뿐이려면 단조성(측도가능성)이 필요하다.

**증명 (아이디어 한 줄, 인용).** $\bar F(s+t)=\bar F(s)\bar F(t)$에서 $\bar F(m/n)=\bar F(1)^{m/n}$ (유리수), 단조성으로 모든 실수 $t$로 확장해 $\bar F(t)=e^{-\lambda t}$, $\lambda=-\log\bar F(1)$. 자세한 것은 Ross 12e §5.2.2.

### 2.7 직관

$E[X\mid Y]$는 "$Y$를 알고 난 뒤 $X$의 최선 예측"이고, 데이터로 보면 **회귀곡선**이다: $Y$를 구간으로 나눠 각 구간의 $X$ 평균을 찍으면 그 곡선이 나온다. 탑 성질은 "구간 평균들의 가중평균 = 전체 평균"이라는 당연한 말이다.

랜덤 합의 분산은 "몇 개인가($N$)"와 "각각 얼마나 큰가($Y$)" 두 요인의 합이고, 푸아송은 평균 = 분산이라 두 항이 $\lambda E[Y^2]$ 하나로 합쳐진다.

지수 시계 경주는 "율(rate)은 더해지고, 누가 이기는지는 율에 비례하며, 언제 끝나는지와 무관하다" — 이것이 CTMC(03a)의 생성원 $Q$의 각 행이 뜻하는 전부다.

### 2.8 함정(traps)

1. $E[X\mid Y]$는 숫자가 아니라 **확률변수**($Y$의 함수)다. $E[X\mid Y=y]$가 숫자. 두 표기를 섞으면 탑 성질을 잘못 적용한다.
2. $\mathrm{Var}(S)=E[N]\mathrm{Var}(Y)$라고 쓰는 실수: $\mathrm{Var}(N)(EY)^2$ 항이 빠졌다. 같은 $E[N]=3$이라도 $N\sim$Poisson(3)이면 7.5, $N\sim$Bin(6, 1/2)이면 4.125.
3. `rng.exponential(2.0)`은 rate 2가 아니라 **scale 2**(평균 2, rate 1/2). `scipy.stats.expon(scale=...)`도 같다. 항상 `scale=1/rate`로 적는다.
4. 구간화 추정은 구간 안에서 $E[X\mid Y]$가 변하므로 **편향**이 있다: 구간 $(1,1.25)$의 참값은 $1.1198$이지 중심 $1.125$가 아니다. 구간을 좁히면 편향은 줄고 분산은 는다.
5. pgf는 $s^X$의 기댓값, mgf는 $e^{tX}$의 기댓값: $G(s)=M(\log s)$. $G'(1)$은 평균이지만 $G''(1)$은 분산이 아니라 $E[X(X-1)]$.
6. 지수분포의 최솟값은 지수분포지만 **최댓값은 아니다**. 또 무기억성이 있으면 "오래 기다렸으니 곧 온다"는 직관이 틀린다: $E[X-s\mid X>s]=1/\lambda$. 균등분포 $U(0,b)$라면 $(b-s)/2$로 줄어든다.

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요)

- 정리 B의 증명을 안 보고 다시 써 보세요. 어느 줄에서 "$N$과 $Y$의 독립"이 쓰이나요?
- 정리 D(iii)의 적분 한 줄을 다시 써 보세요. 어디에서 곱의 꼴이 나오나요?

## 3. Predict — 코드를 돌리기 전에 예측

아래 셀의 `None`을 숫자로 바꿔 보세요. 정확할 필요는 없습니다 — 자릿수와 방향(기준값보다 위/아래)이 맞는지가 목적입니다. 예측을 먼저 적어 두고 4절을 실행하세요.

In [ ]:
predictions = {
    # Y ~ Exp(1), X | Y ~ Poisson(Y) 일 때 P(X = 0) 은? (e^{-1} = 0.368 일까?)
    "p_x_zero": None,
    # 같은 X 의 분산은? (Poisson(1) 이면 1 이다.)
    "var_x": None,
    # Y_i in {1, 2} 등확률, N ~ Poisson(3) 일 때 S = sum_{i<=N} Y_i 의 분산은? (E[S] = 4.5)
    "var_s_poisson": None,
    # 같은 Y_i, N ~ Bin(6, 1/2) (E[N] = 3 동일) 일 때 Var(S) 는 7.5 보다 클까 작을까?
    "var_s_binomial": None,
    # 독립 지수 시계 세 개 (율 1, 2, 3) 중 가장 느린 시계 (율 1) 가 가장 먼저 울릴 확률은?
    "p_clock1_wins": None,
    # '율 1 시계가 이겼다' 는 조건 하에 첫 울림 시각의 기댓값은? 무조건 기댓값 1/6 보다 클까?
    "e_min_given_clock1": None,
}

## 4. Simulate — 시뮬레이션

### 4.1 예제 A — 조건부 기대의 두 방향

예제 A: $Y\sim\mathrm{Exp}(1)$, $X\mid Y\sim\mathrm{Poisson}(Y)$ ("율이 랜덤인 푸아송"). 알려진 것은 $E[X\mid Y=y]=y$, 그리고 (E2에서 유도) 반대 방향 $E[Y\mid X=k]=(k+1)/2$다. 데이터에서 조건부 기대를 추정하는 방법은 두 가지다: 연속 $Y$는 **구간화(binning)** 해서 구간별 $X$ 평균, 이산 $X$는 **그룹 평균** `Y[X == k].mean()`.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def binned_mean(y, x, edges):
    """y 를 edges 로 구간화해 각 구간의 x 평균·개수·SE 를 돌려준다 (범위 밖은 제외)."""
    idx = np.digitize(y, edges) - 1
    ok = (idx >= 0) & (idx < len(edges) - 1)
    idx, x = idx[ok], np.asarray(x, dtype=float)[ok]
    counts = np.bincount(idx, minlength=len(edges) - 1)
    safe = np.maximum(counts, 1)
    means = np.bincount(idx, weights=x, minlength=len(edges) - 1) / safe
    var_within = np.bincount(idx, weights=x**2, minlength=len(edges) - 1) / safe - means**2
    centers = (edges[:-1] + edges[1:]) / 2
    return centers, means, counts, np.sqrt(np.maximum(var_within, 0) / safe)

def random_sum(N, jump_sampler, rng):
    """S_j = sum of N_j i.i.d. jumps: 점프를 한 번에 뽑고 누적합 차분 (N=0 인 경로는 0)."""
    J = jump_sampler(int(N.sum()), rng)
    cs = np.concatenate([[0.0], np.cumsum(J)])
    ends = np.cumsum(N)
    return cs[ends] - cs[ends - N]

def pmf_from_pgf(G, M=128):
    """단위원 위 M 개 점에서 pgf 를 평가해 FFT 로 pmf 를 복원한다 (지지집합 << M 이면 정확)."""
    z = np.exp(2j * np.pi * np.arange(M) / M)
    return np.clip(np.fft.fft(G(z)).real / M, 0.0, None)

In [ ]:
mu_Y = 1.0                                          # Y ~ Exp(1): scale = mean = 1
Y = rng.exponential(mu_Y, n)                        # scale=1 -> rate 1
X = rng.poisson(Y)                                  # 율이 Y 인 푸아송

edges = np.arange(0, 5.001, 0.25)
centers, means, counts, ses = binned_mean(Y, X, edges)
ks = np.arange(0, 7)
group_means = np.array([Y[X == k].mean() for k in ks])
group_ses = np.array([Y[X == k].std(ddof=1) / np.sqrt((X == k).sum()) for k in ks])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.scatter(Y[:1500], X[:1500], s=4, alpha=0.25, color="gray", label="sample (Y, X), first 1500")
ax1.errorbar(centers, means, yerr=1.96 * ses, fmt="o", ms=4, capsize=2, label="binned mean of X | Y in bin")
ax1.plot(edges, edges, "C1-", label="exact E[X | Y=y] = y")
ax1.set_xlabel("Y"); ax1.set_ylabel("X"); ax1.set_ylim(-0.5, 10)
ax1.set_title("(a) E[X | Y]: binned means (bin width 0.25)"); ax1.legend(loc="upper left", fontsize=8)
ax2.errorbar(ks, group_means, yerr=1.96 * group_ses, fmt="o", ms=5, capsize=3, label="mean of Y | X = k")
ax2.plot(ks, (ks + 1) / 2, "C1-", label="exact (k+1)/2")
ax2.set_xlabel("k"); ax2.set_ylabel("Y"); ax2.set_title("(b) E[Y | X = k]"); ax2.legend(loc="upper left")
plt.show()

두 방향 모두 직선 위에 얹힌다. (a)의 오른쪽 끝은 표본이 적어 오차막대가 커진다($P(Y>4.75)\approx0.9\%$). 구간 안에서 참값 $y$가 변하므로 구간 평균은 구간 **중심**이 아니라 구간 안의 $Y$의 조건부 평균을 추정한다(함정 4). 구간 $(1,1.25)$에서 정확히 계산하면
$$E[X\mid 1<Y<1.25]=E[Y\mid 1<Y<1.25]=1+\frac{e^{-1}-1.25e^{-1.25}}{e^{-1}-e^{-1.25}}=1.1198,$$
중심 $1.125$보다 작다 — 지수 밀도가 구간 안에서 감소하기 때문이다.

In [ ]:
in_bin = (Y > 1) & (Y < 1.25)
est_bin = mc_estimate(X[in_bin])
exact_bin = 1 + (np.exp(-1) - 1.25 * np.exp(-1.25)) / (np.exp(-1) - np.exp(-1.25))
est_y_given_x2 = mc_estimate(Y[X == 2])
exact_y_given_x2 = (2 + 1) / 2
print(f"E[X | 1<Y<1.25]: MC {est_bin}   exact {exact_bin:.6f}   (bin center 1.125)   "
      f"z = {(est_bin.mean - exact_bin) / est_bin.se:+.2f}")
print(f"E[Y | X=2]     : MC {est_y_given_x2}   exact {exact_y_given_x2:.4f}   "
      f"z = {(est_y_given_x2.mean - exact_y_given_x2) / est_y_given_x2.se:+.2f}")

### 4.2 탑 성질과 분산 분해를 숫자로

탑 성질 $E[E[X\mid Y]]=E[X]$는 데이터에서 **항등식**이다: 구간 평균들의 개수 가중평균은 (모든 표본을 빠짐없이 덮는 분할이면) 전체 평균과 부동소수 오차 안에서 같다. 예제 A의 정확값: $E[X]=E[Y]=1$, $\mathrm{Var}X=E[\mathrm{Var}(X\mid Y)]+\mathrm{Var}(E[X\mid Y])=E[Y]+\mathrm{Var}(Y)=1+1=2$, $P(X=0)=E[e^{-Y}]=\int_0^\infty e^{-y}e^{-y}dy=1/2$ (푸아송(1)이라면 $e^{-1}=0.368$).

In [ ]:
edges_all = np.append(edges, np.inf)                # 모든 표본을 덮는 분할 (마지막 구간 = (5, inf))
_, m_all, c_all, se_all = binned_mean(Y, X, edges_all)
tower_lhs = (c_all * m_all).sum() / c_all.sum()
np.testing.assert_allclose(tower_lhs, X.mean(), atol=1e-9)   # 항등식
print(f"weighted mean of bin means = {tower_lhs:.6f} == X.mean() = {X.mean():.6f}   (tower, exact identity)")

est_ex = mc_estimate(X)
est_varx = mc_estimate((X - X.mean()) ** 2)
est_p0 = mc_proportion(X == 0)
e_var_within = (c_all * se_all**2 * c_all).sum() / c_all.sum()      # E[Var(X|Y)] 의 구간화 근사
var_of_means = (c_all * (m_all - X.mean()) ** 2).sum() / c_all.sum()  # Var(E[X|Y]) 의 구간화 근사
exact_ex = mu_Y                                    # E[X] = E[Y]
exact_varx = mu_Y + mu_Y**2                        # E[Y] + Var Y
exact_p0 = 1 / (1 + mu_Y)                          # E[e^{-Y}] = (1/mu)/(1/mu + 1)
print(f"E[X]   : MC {est_ex}   exact {exact_ex}")
print(f"Var X  : MC {est_varx}   exact {exact_varx}")
print(f"P(X=0) : MC {est_p0}   exact {exact_p0}")
print(f"decomposition (binned): E[Var(X|Y)] ~ {e_var_within:.4f} (exact E[Y] = 1),  "
      f"Var(E[X|Y]) ~ {var_of_means:.4f} (exact Var Y = 1),  sum = {e_var_within + var_of_means:.4f}")

분산 2는 푸아송(1)의 분산 1의 두 배다: 율 자체의 흔들림($\mathrm{Var}Y=1$)이 그대로 더해졌다. 두 분해 항은 구간화 근사라 표만 보고 넘어간다(구간 안에서 $E[X\mid Y]$가 변하는 부분이 첫 항에 조금 새어 들어간다).

### 4.3 랜덤 합 — 같은 평균, 다른 퍼짐

점프 $Y_i\in\{1,2\}$ 등확률. (a) $N\sim\mathrm{Poisson}(3)$: `spkit.poisson.compound_poisson`이 경로마다 $N$을 뽑고 점프를 합친다. (b) $N\sim\mathrm{Bin}(6,\tfrac12)$: 같은 $E[N]=3$이지만 $\mathrm{Var}N=1.5<3$. 정리 B: $E[S]=4.5$ 둘 다, $\mathrm{Var}S=7.5$ vs $4.125$.

In [ ]:
jv, jp = np.array([1.0, 2.0]), np.array([0.5, 0.5])  # 점프 값과 확률
jumps = lambda m, g: g.integers(1, 3, size=m)       # {1, 2} 등확률 (jv 에서 jp 로 뽑기)
lam = 3.0                                            # N ~ Poisson(lam): E[N] = Var N = lam
nB, pB = 6, 0.5                                      # N ~ Bin(nB, pB): E[N] = 3 (같음), Var N = 1.5
S_pois = compound_poisson(rate=lam, T=1.0, jump_sampler=jumps, rng=rng, n_paths=n)
N_b = rng.binomial(nB, pB, n)
S_bin = random_sum(N_b, jumps, rng)

EY, EY2 = (jv * jp).sum(), (jv**2 * jp).sum()
VarY = EY2 - EY**2
EN_b, VarN_b = nB * pB, nB * pB * (1 - pB)
exact_es = lam * EY                                  # E[N] E[Y]  (EN_b == lam 이라 두 경우 같다)
exact_vs_pois = lam * EY2                            # lambda E[Y^2]
exact_vs_bin = EN_b * VarY + VarN_b * EY**2          # E[N] Var Y + Var N (EY)^2
est_es_pois, est_vs_pois = mc_estimate(S_pois), mc_estimate((S_pois - S_pois.mean()) ** 2)
est_es_bin, est_vs_bin = mc_estimate(S_bin), mc_estimate((S_bin - S_bin.mean()) ** 2)
print(f"Poisson(3) N : E[S] {est_es_pois}  (exact {exact_es}),   Var S {est_vs_pois}  (exact {exact_vs_pois})")
print(f"Bin(6,1/2) N : E[S] {est_es_bin}  (exact {exact_es}),   Var S {est_vs_bin}  (exact {exact_vs_bin})")

In [ ]:
fig, ax = plt.subplots()
bins = np.arange(-0.5, 25.5)
ax.hist(S_pois, bins=bins, density=True, alpha=0.5, label=f"N ~ Poisson(3): Var S = {exact_vs_pois}")
ax.hist(S_bin, bins=bins, density=True, alpha=0.5, label=f"N ~ Binomial(6, 1/2): Var S = {exact_vs_bin}")
ax.axvline(exact_es, color="k", ls="--", lw=1, label=f"common mean {exact_es}")
ax.set_xlabel("S = sum of N jumps in {1, 2}"); ax.set_ylabel("density")
ax.set_title("Random sums with the same E[N] = 3 but different Var(N)"); ax.legend()
plt.show()

이항 $N$은 $N\le6$이라 $S\le12$로 잘리고 푸아송 $N$은 긴 오른쪽 꼬리를 가진다. 두 히스토그램의 평균은 같지만, 분산의 차이 $7.5-4.125=3.375=(3-1.5)\cdot1.5^2$는 정확히 $\mathrm{Var}N$의 차이에 $(EY)^2$을 곱한 것이다(함정 2).

### 4.4 pgf에서 pmf 뽑기 (정리 C + FFT)

$G_Y(s)=(s+s^2)/2$, $G_N(s)=e^{3(s-1)}$이므로 $G_S=G_N\circ G_Y$. pmf를 손으로 전개하는 대신 단위원 위 $M=128$개 점에서 $G_S$를 평가하고 FFT로 계수를 읽는다. 검사: 합 1, 평균 4.5, 분산 7.5 (수치 항등식, $10^{-8}$). 손 계산 한 개: $P(S=2)$는 "$N=1,Y_1=2$" 또는 "$N=2,Y=(1,1)$"이므로 $e^{-3}\bigl(3\cdot\tfrac12+\tfrac92\cdot\tfrac14\bigr)=e^{-3}(1.5+1.125)=0.130691$.

In [ ]:
G_Y = lambda s: (s + s**2) / 2
G_N = lambda s: np.exp(lam * (s - 1))
G_S = lambda s: G_N(G_Y(s))
G_Sb = lambda s: (1 - pB + pB * G_Y(s)) ** nB       # N ~ Bin(6, 1/2): G_N(s) = (1 - p + p s)^6
M = 128
pmf_S, pmf_Sb = pmf_from_pgf(G_S, M), pmf_from_pgf(G_Sb, M)
k = np.arange(M)
for name, pmf, v in (("Poisson", pmf_S, exact_vs_pois), ("Binomial", pmf_Sb, exact_vs_bin)):
    mean_k, var_k = (k * pmf).sum(), (k**2 * pmf).sum() - (k * pmf).sum() ** 2
    np.testing.assert_allclose([pmf.sum(), mean_k, var_k], [1.0, exact_es, v], atol=1e-8)
    print(f"{name:8s}: sum {pmf.sum():.10f}, mean {mean_k:.6f}, var {var_k:.6f}, pmf[:5] = {np.round(pmf[:5], 6)}")

exact_p2 = np.exp(-lam) * (lam * jp[1] + lam**2 / 2 * jp[0] ** 2)   # N=1,Y=2 또는 N=2,Y=(1,1)
np.testing.assert_allclose(pmf_S[2], exact_p2, atol=1e-10)
est_p2 = mc_proportion(S_pois == 2)
print(f"P(S=2): pgf/FFT {pmf_S[2]:.7f}, hand {exact_p2:.7f}, MC {est_p2}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
plot_hist_vs_pmf(S_pois, np.arange(30), pmf_S[:30], ax=ax1, label_pmf="pmf from G_N(G_Y(s)) by FFT")
ax1.set_title("N ~ Poisson(3), jumps in {1, 2}"); ax1.set_xlabel("S"); ax1.set_xlim(-0.5, 25)
plot_hist_vs_pmf(S_bin, np.arange(14), pmf_Sb[:14], ax=ax2, label_pmf="pmf from ((1+G_Y(s))/2)^6 by FFT")
ax2.set_title("N ~ Binomial(6, 1/2), jumps in {1, 2}"); ax2.set_xlabel("S"); ax2.legend(loc="upper right")
plt.show()

pgf 하나가 pmf 전체·평균·분산을 다 준다. FFT 복원이 정확한 이유: $G(e^{i\theta})=\sum_kp_ke^{ik\theta}$는 푸리에 급수이고, 지지집합이 $M$보다 작으면 $M$개 점의 이산 푸리에 변환이 계수를 정확히 되돌린다(길이 $M$을 넘는 꼬리는 $k\bmod M$으로 접히지만 여기서는 $10^{-16}$ 수준).

### 4.5 무기억성 — 지수 vs 균등

$X\sim\mathrm{Exp}(\text{rate }0.5)$(평균 2)를 `rng.exponential(scale=2.0, ...)`로 뽑는다(함정 3: 인수 이름을 명시한다). $X>3$인 표본의 잔여수명 $X-3$은 다시 $\mathrm{Exp}(0.5)$: 평균 2, 3만큼 기다린 것이 아무 정보도 아니다. 대조: $U\sim\mathrm{Uniform}(0,4)$(평균 2)는 $U>3$이면 잔여수명이 $\mathrm{Uniform}(0,1)$, 평균 0.5 — 오래 기다릴수록 곧 온다.

In [ ]:
rate, b = 0.5, 4.0                                   # Exp(rate 0.5) 와 Uniform(0, 4): 둘 다 평균 2
X_exp = rng.exponential(scale=1 / rate, size=n)     # (scale = 1/rate !)
U = rng.uniform(0, b, n)
s0 = 3.0
resid = X_exp[X_exp > s0] - s0
resid_u = U[U > s0] - s0
est_resid, est_resid_u = mc_estimate(resid), mc_estimate(resid_u)
exact_resid, exact_resid_u = 1 / rate, (b - s0) / 2
print(f"Exp(rate 0.5): P(X>3) = {np.mean(X_exp > s0):.4f} (exact e^-1.5 = {np.exp(-rate * s0):.4f});  "
      f"E[X-3 | X>3] MC {est_resid}  exact {exact_resid}")
print(f"Uniform(0,4) : P(U>3) = {np.mean(U > s0):.4f} (exact {(b - s0) / b:.2f});  "
      f"E[U-3 | U>3] MC {est_resid_u}  exact {exact_resid_u}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
x = np.linspace(0, 12, 300)
plot_hist_vs_pdf(resid, x, rate * np.exp(-rate * x), ax=ax1, bins=60, label_pdf="Exp(rate 0.5) density (same as X)")
ax1.set_title("Residual life X - 3 given X > 3, X ~ Exp(rate 0.5)"); ax1.set_xlabel("X - 3")
xu = np.linspace(0, 1, 100)
plot_hist_vs_pdf(resid_u, xu, np.ones_like(xu), ax=ax2, bins=20, label_pdf="Uniform(0, 1) density")
ax2.set_title("Residual life U - 3 given U > 3, U ~ Uniform(0, 4)"); ax2.set_xlabel("U - 3")
plt.show()

왼쪽 히스토그램은 조건 없는 $X$의 밀도와 **똑같다** — 지수분포에서는 "기다린 시간"이 정보가 아니다. 오른쪽은 남은 시간이 1 이하로 잘려 있다 — 균등분포에서는 오래 기다릴수록 곧 온다. 정리 E: 연속 분포 중 왼쪽 그림이 나오는 것은 지수분포뿐이다.

### 4.6 경쟁하는 시계

율 $(1,2,3)$의 독립 지수 시계 세 개, $\Lambda=6$. 정리 D: 첫 울림 $M\sim\mathrm{Exp}(6)$ (평균 $1/6$), 승자 확률 $(1,2,3)/6$, 그리고 승자와 시각이 독립 — "느린 시계(율 1)가 이겼으니 오래 걸렸을 것"이라는 직관은 틀리고 $E[M\mid I=1]=E[M]=1/6$이다 (코드에서는 0부터 세므로 `I == 0`이 율 1 시계다).

In [ ]:
rates = np.array([1.0, 2.0, 3.0])
Lam = rates.sum()
T3 = rng.exponential(scale=1 / rates, size=(n, 3))   # 브로드캐스트: 열마다 다른 scale
Mn = T3.min(axis=1)
I = T3.argmin(axis=1)

est_win = [mc_proportion(I == i) for i in range(3)]
exact_win = rates / Lam
est_min = mc_estimate(Mn)
est_min_given = [mc_estimate(Mn[I == i]) for i in range(3)]
exact_min = 1 / Lam
for i in range(3):
    print(f"clock {i + 1} (rate {rates[i]:.0f}): P(wins) MC {est_win[i]}  exact {exact_win[i]:.4f};   "
          f"E[min | clock {i + 1} wins] MC {est_min_given[i]}")
print(f"E[min] MC {est_min}   exact 1/6 = {exact_min:.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.bar(np.arange(3) - 0.18, [e.mean for e in est_win], width=0.36, yerr=[1.96 * e.se for e in est_win],
        capsize=3, label="MC winner frequency")
ax1.bar(np.arange(3) + 0.18, exact_win, width=0.36, alpha=0.6, label="exact rate_i / sum")
ax1.set_xticks(range(3), ["rate 1", "rate 2", "rate 3"]); ax1.set_ylabel("P(clock i rings first)")
ax1.set_title("(a) Who wins the race"); ax1.legend()
xm = np.linspace(0, 1.2, 300)
hb = np.linspace(0, 1.2, 49)
for i in range(3):
    ax2.hist(Mn[I == i], bins=hb, density=True, alpha=0.4, label=f"min | winner = rate {rates[i]:.0f}")
ax2.plot(xm, Lam * np.exp(-Lam * xm), "k-", lw=2, label="Exp(6) density")
ax2.set_xlabel("first ring time M"); ax2.set_ylabel("density"); ax2.set_title("(b) Min time given the winner: all Exp(6)")
ax2.legend(fontsize=8)
plt.show()

(b)의 세 히스토그램이 모두 같은 $\mathrm{Exp}(6)$ 곡선 위에 얹힌다 — 승자를 알아도 시각의 분포는 바뀌지 않는다(정리 D(iii)). 이 사실이 03a에서 "CTMC는 $\mathrm{Exp}(q_i)$만큼 머문 뒤 $q_{ij}/q_i$로 점프한다"는 두 문장으로 쪼개지는 근거다.

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

In [ ]:
rows = [
    {"name": "E[X] (tower)", "predicted": None, "estimate": est_ex, "exact": exact_ex},
    {"name": "Var X = E[Y] + Var Y", "predicted": predictions["var_x"], "estimate": est_varx, "exact": exact_varx},
    {"name": "P(X = 0)", "predicted": predictions["p_x_zero"], "estimate": est_p0, "exact": exact_p0},
    {"name": "E[X | 1 < Y < 1.25] (not the bin center 1.125)", "predicted": None, "estimate": est_bin, "exact": exact_bin},
    {"name": "E[Y | X = 2]", "predicted": None, "estimate": est_y_given_x2, "exact": exact_y_given_x2},
    {"name": "E[S], N ~ Poisson(3)", "predicted": None, "estimate": est_es_pois, "exact": exact_es},
    {"name": "Var S, N ~ Poisson(3)", "predicted": predictions["var_s_poisson"], "estimate": est_vs_pois, "exact": exact_vs_pois},
    {"name": "Var S, N ~ Binomial(6, 1/2)", "predicted": predictions["var_s_binomial"], "estimate": est_vs_bin, "exact": exact_vs_bin},
    {"name": "P(S = 2), N ~ Poisson(3) (pgf/FFT)", "predicted": None, "estimate": est_p2, "exact": float(pmf_S[2])},
    {"name": "E[X - 3 | X > 3], X ~ Exp(rate 0.5)", "predicted": None, "estimate": est_resid, "exact": exact_resid},
    {"name": "E[U - 3 | U > 3], U ~ Uniform(0, 4)", "predicted": None, "estimate": est_resid_u, "exact": exact_resid_u},
    {"name": "P(rate-1 clock wins)", "predicted": predictions["p_clock1_wins"], "estimate": est_win[0], "exact": float(exact_win[0])},
    {"name": "E[min]", "predicted": None, "estimate": est_min, "exact": exact_min},
    {"name": "E[min | rate-1 clock wins]", "predicted": predictions["e_min_given_clock1"], "estimate": est_min_given[0], "exact": exact_min},
]
Markdown(compare_table(rows))

각 행에서 "investigate"가 뜬다면:
- **E[X] (tower)**: `rng.poisson(Y)`에 벡터 $Y$가 아니라 스칼라가 들어갔거나 $Y$의 scale이 1이 아니다.
- **Var X**: 분산이 1 근처면 $Y$의 흔들림이 빠진 것(모든 $X$가 같은 율) — $\mathrm{Var}(E[X\mid Y])$ 항이 사라진 셈이다.
- **P(X = 0)**: 0.368이 나오면 위와 같은 원인. $E[e^{-Y}]=1/2$는 $Y$가 랜덤이어야 나온다.
- **E[X | 1<Y<1.25]**: 구간 중심 1.125와 비교했다면 (n이 크면) investigate가 뜬다 — 그것이 함정 4다. 부등호 방향·경계도 확인.
- **E[Y | X = 2]**: `Y[X == 2]`의 마스크가 어긋났거나(예: `X == 2.0` 부동소수) 사후분포 공식이 틀렸다.
- **E[S] Poisson**: `compound_poisson`의 `rate*T`가 3이 아니거나 점프 표본기가 `{1,2}`가 아니라 `{1}`·`{0,1}`을 낸다(`integers`의 상한은 배타적).
- **Var S Poisson / Binomial**: 두 값이 서로 바뀌었거나 $\mathrm{Var}N(EY)^2$ 항을 빼먹었다(함정 2). `random_sum`의 인덱스 `ends - N`이 틀리면 평균부터 어긋난다.
- **P(S = 2)**: FFT의 부호(`fft` vs `ifft`)나 $M$이 지지집합보다 작다.
- **memoryless residual**: `rng.exponential(0.5)`라고 썼다면 평균 0.5짜리를 뽑은 것(함정 3) — 잔여 평균이 0.5로 나온다.
- **uniform residual**: 조건 `U > 3`의 방향 또는 상한 4가 틀렸다.
- **P(rate-1 clock wins)**: `argmin` 대신 `argmax`, 또는 `scale=rates`(rate와 scale 뒤바뀜)라면 승자 확률이 $(3,2,1)/6$ 순으로 뒤집힌다.
- **E[min]**, **E[min | rate-1 wins]**: 둘 중 하나만 어긋나면 독립성이 깨진 것 — 시계에 공통 난수를 섞었거나 균등 시계를 썼을 때 그렇다.

In [ ]:
assert_close(est_ex, exact_ex, k=4, name="E[X]")
assert_close(est_varx, exact_varx, k=4, name="Var X")
assert_close(est_p0, exact_p0, k=4, name="P(X=0)")
assert_close(est_bin, exact_bin, k=4, name="E[X | 1<Y<1.25]")
assert_close(est_y_given_x2, exact_y_given_x2, k=4, name="E[Y | X=2]")
assert_close(est_es_pois, exact_es, k=4, name="E[S] Poisson")
assert_close(est_vs_pois, exact_vs_pois, k=4, name="Var S Poisson")
assert_close(est_vs_bin, exact_vs_bin, k=4, name="Var S Binomial")
assert_close(est_p2, float(pmf_S[2]), k=4, name="P(S=2) via pgf")
assert_close(est_resid, exact_resid, k=4, name="memoryless residual")
assert_close(est_resid_u, exact_resid_u, k=4, name="uniform residual")
assert_close(est_win[0], float(exact_win[0]), k=4, name="P(clock 1 wins)")
assert_close(est_min, exact_min, k=4, name="E[min]")
assert_close(est_min_given[0], exact_min, k=4, name="E[min | clock 1 wins]")
print("all assert_close(k=4) checks passed")

## 6. 연습문제

### E1 계산

$N\sim\mathrm{Poisson}(4)$, $Y_i\in\{0,1,2\}$ 확률 $(1/4,1/2,1/4)$, 서로 독립.
(a) $E[S]$, $\mathrm{Var}(S)$. (b) $G_Y(s)$와 $G_S(s)$를 쓰고 $P(S=0)$을 구하라. (c) 독립 지수 시계 율 $(1,1,2)$에서 율 2 시계가 이길 확률과 $E[\min]$. (d) $X\sim\mathrm{Exp}(\lambda)$에 대해 $E[X\mid X>s]$.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) $E[Y]=0\cdot\tfrac14+1\cdot\tfrac12+2\cdot\tfrac14=1$, $E[Y^2]=0+\tfrac12+4\cdot\tfrac14=1.5$. 복합 푸아송이므로 $E[S]=\lambda E[Y]=4$, $\mathrm{Var}S=\lambda E[Y^2]=4\cdot1.5=6$.

(b) $G_Y(s)=\tfrac14+\tfrac12s+\tfrac14s^2=\bigl(\tfrac{1+s}2\bigr)^2$ (즉 $Y\sim\mathrm{Bin}(2,\tfrac12)$). $G_S(s)=G_N(G_Y(s))=\exp\bigl(4\bigl[(\tfrac{1+s}2)^2-1\bigr]\bigr)$. $P(S=0)=G_S(0)=e^{4(1/4-1)}=e^{-3}=0.049787$. (직접 확인: $S=0\iff$ $N$개의 $Y$가 모두 0, 즉 $\sum_nP(N=n)(1/4)^n=e^{-4}e^{4/4}=e^{-3}$.)

(c) $\Lambda=4$: $P(\text{율 2 시계 승})=2/4=1/2$, $E[\min]=1/4$.

(d) 무기억성으로 $(X-s)\mid X>s\sim\mathrm{Exp}(\lambda)$이므로 $E[X\mid X>s]=s+1/\lambda$.

pgf/FFT로 (a)(b)를 한 번에 검산:

```python
G_Y1 = lambda s: ((1 + s) / 2) ** 2
G_S1 = lambda s: np.exp(4 * (G_Y1(s) - 1))
pmf1 = pmf_from_pgf(G_S1, 128); k = np.arange(128)
print(pmf1[0], np.exp(-3))                                   # 0.049787 둘 다
print((k * pmf1).sum(), (k**2 * pmf1).sum() - (k * pmf1).sum() ** 2)   # 4.0, 6.0
S1 = compound_poisson(4.0, 1.0, lambda m, g: g.choice([0, 1, 2], size=m, p=[0.25, 0.5, 0.25]), rng, n_paths=n)
print(mc_estimate(S1), mc_estimate((S1 - S1.mean()) ** 2), mc_proportion(S1 == 0))
```

</details>

### E2 유도 후 시뮬레이션 검증

$Y\sim\mathrm{Exp}(1)$, $X\mid Y\sim\mathrm{Poisson}(Y)$에 대해 (i) 주변 pmf $P(X=k)=2^{-(k+1)}$, (ii) 사후분포 $Y\mid X=k\sim\mathrm{Gamma}(k+1,\ \text{rate }2)$, 따라서 $E[Y\mid X=k]=(k+1)/2$, $\mathrm{Var}(Y\mid X=k)=(k+1)/4$를 유도하라. 그다음 4.1의 `X, Y`로 $k=0,\dots,5$에 대해 `Y[X == k]`의 평균·분산을 정확값과 대조하고, $k=3$에서 `plot_hist_vs_pdf`로 $\mathrm{Gamma}(4,\ \text{scale }1/2)$ 밀도와 겹쳐라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(i) 조건화로 확률을 계산한다(Ross 12e §3.5):
$$P(X=k)=\int_0^\infty e^{-y}\,\frac{y^ke^{-y}}{k!}\,dy=\frac1{k!}\int_0^\infty y^ke^{-2y}\,dy=\frac1{k!}\cdot\frac{k!}{2^{k+1}}=2^{-(k+1)}$$
($\int_0^\infty y^ke^{-cy}dy=k!/c^{k+1}$). 즉 $X$는 $\{0,1,2,\dots\}$ 위의 기하분포($p=1/2$): $P(X=0)=1/2$, $E[X]=1$, $\mathrm{Var}X=2$ — 4.2의 숫자들이다.

(ii) 베이즈: $f(y\mid k)\propto f_Y(y)\,P(X=k\mid Y=y)\propto e^{-y}y^ke^{-y}=y^ke^{-2y}$. 이것은 $\mathrm{Gamma}(\text{shape }k+1,\ \text{rate }2)$의 핵이다. 감마의 평균 shape/rate $=(k+1)/2$, 분산 shape/rate$^2=(k+1)/4$. $k=3$: $E[Y\mid X=3]=2$, $\mathrm{Var}=1$, $P(X=3)=1/16$.

```python
for kk in range(6):
    sel = Y[X == kk]
    est_m = mc_estimate(sel)
    est_v = mc_estimate((sel - sel.mean()) ** 2)
    assert_close(est_m, (kk + 1) / 2, k=4, name=f"E[Y|X={kk}]")
    assert_close(est_v, (kk + 1) / 4, k=4, name=f"Var(Y|X={kk})")
    print(f"k={kk}: P(X=k) {mc_proportion(X == kk)} (exact {2.0**-(kk+1):.5f});  "
          f"mean {est_m} (exact {(kk+1)/2});  var {est_v} (exact {(kk+1)/4})")
yy = np.linspace(0, 8, 300)
ax = plot_hist_vs_pdf(Y[X == 3], yy, gamma(a=4, scale=0.5).pdf(yy), bins=60,
                      label_pdf="Gamma(shape 4, rate 2) density")
ax.set_xlabel("Y"); ax.set_title("Posterior of the rate Y given X = 3"); plt.show()
```

</details>

### E3 가정을 깨보기

정리 B의 "$N$과 $Y$의 독립"을 깨자. $Y_i\in\{1,2\}$ 등확률.
(a) **정지 시간형**: $N=1$ if $Y_1=2$ else $2$ (첫 점프가 크면 멈춤 — 과거만 본다).
(b) **미래 엿보기형**: $N=2$ if $Y_2=2$ else $1$ (두 번째 점프를 미리 보고 크면 포함).
각각 $E[S]$와 $\mathrm{Var}(S)$를 시뮬레이션하고 순진한 공식 $E[N]E[Y]$, $E[N]\mathrm{Var}Y+\mathrm{Var}N(EY)^2$와 비교하라. 어느 쪽이, 어느 공식이 살아남는가?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

두 경우 모두 $N\in\{1,2\}$ 등확률이라 $E[N]=1.5$, $\mathrm{Var}N=0.25$이고, 순진한 공식은 $E[S]=1.5\cdot1.5=2.25$, $\mathrm{Var}S=1.5\cdot0.25+0.25\cdot2.25=0.9375$를 준다.

(a) $Y_1=2$이면 $S=2$; $Y_1=1$이면 $S=1+Y_2\in\{2,3\}$ 등확률. 따라서 $S=2$ (확률 $3/4$), $S=3$ ($1/4$): $E[S]=2.25$ — **평균 공식은 살아남는다**($N$이 정지 시간이면 성립하는 Wald 항등식, 05c에서 증명). 그러나 $\mathrm{Var}S=\tfrac34\cdot\tfrac14=0.1875\ne0.9375$ — **분산 공식은 깨진다**. 큰 점프가 나오면 멈추는 규칙이 합의 흔들림을 눌러 버렸기 때문이다.

(b) $S=Y_1+2\cdot\mathbf 1\{Y_2=2\}$: $E[S]=1.5+1=2.5\ne2.25$ — 미래를 엿보면 **평균 공식마저 깨진다**($S$에 포함되는 $Y_2$는 항상 2라 $E[Y]$가 아니다). $\mathrm{Var}S=0.25+1=1.25$.

```python
Y1, Y2 = rng.integers(1, 3, n), rng.integers(1, 3, n)
N_a = np.where(Y1 == 2, 1, 2); S_a = np.where(Y1 == 2, Y1, Y1 + Y2)      # (a) 과거만 본다
N_b2 = np.where(Y2 == 2, 2, 1); S_b = np.where(Y2 == 2, Y1 + Y2, Y1)     # (b) 미래를 본다
naive_mean = 1.5 * 1.5
naive_var = 1.5 * 0.25 + 0.25 * 1.5**2
for tag, S_, e_m, e_v in (("(a) stopping-time", S_a, 2.25, 0.1875), ("(b) peeking", S_b, 2.5, 1.25)):
    em, ev = mc_estimate(S_), mc_estimate((S_ - S_.mean()) ** 2)
    assert_close(em, e_m, k=4, name=tag + " mean"); assert_close(ev, e_v, k=4, name=tag + " var")
    print(f"{tag}: E[S] {em} (true {e_m}, naive {naive_mean});  Var S {ev} (true {e_v}, naive {naive_var})")
```

</details>

## 7. 정리

1. $E[X\mid Y]$는 $Y$의 함수인 **확률변수**(회귀곡선); 구간화·그룹 평균으로 추정하고, 탑 성질 $E[E[X\mid Y]]=E[X]$는 "구간 평균의 가중평균"이라는 데이터 항등식이다.
2. $\mathrm{Var}X=E[\mathrm{Var}(X\mid Y)]+\mathrm{Var}(E[X\mid Y])$: $\mathrm{Exp}(1)$로 섞은 푸아송은 기하분포($P(X=0)=1/2$, 분산 2), 구간 $(1,1.25)$의 조건부 평균은 중심이 아니라 1.1198.
3. 랜덤 합: $E[S]=E[N]E[Y]$, $\mathrm{Var}S=E[N]\mathrm{Var}Y+\mathrm{Var}N(EY)^2$; 푸아송이면 $\lambda E[Y^2]$ (7.5 vs 이항 4.125). $N$이 $Y$에 의존하면 분산 공식부터 깨진다(E3).
4. pgf: $G_S=G_N\circ G_Y$; 단위원 FFT로 pmf를 복원해 히스토그램과 겹칠 수 있다 ($P(S=2)=0.1307$).
5. 지수 시계: 최솟값은 $\mathrm{Exp}(\sum\lambda)$, 승자 확률은 $\lambda_i/\sum\lambda$, 승자와 시각은 독립, 잔여수명은 무기억(균등분포는 아님). numpy의 `exponential` 인수는 `scale=1/rate`.

**trap 카드**
- Q: $N\sim$Poisson(3)과 $N\sim$Bin(6, 1/2)는 평균이 같다. 같은 점프 분포로 만든 랜덤 합 $S$의 분산도 같은가?
- A: 아니다. $\mathrm{Var}S=E[N]\mathrm{Var}Y+\mathrm{Var}N\,(EY)^2$에서 두 번째 항이 다르다: 7.5 vs 4.125. "몇 개인가"의 흔들림이 분산에 그대로 들어간다.

**다음 노트북: 01a** — 조건화(첫걸음 분석의 일반형)와 시뮬레이션 검증 도구가 갖춰졌으므로 이제 마르코프 사슬을 정의할 수 있다. 01a에서 00a의 파산 워크를 흡수 상태 $\{0,N\}$을 가진 사슬로 다시 쓰고, 전이 행렬의 거듭제곱 $P^n$을 시뮬레이션과 대조한다.

**log/progress.md 한 줄 템플릿**

`| YYYY-MM-DD | 00c | __분 | _/5 | (예: Var S 에서 Var N (EY)^2 항 / 승자와 최솟값의 독립) |`